# 02 - CSV para Delta Lake (MinIO)

Lê os arquivos CSV do bucket `landing-zone` no MinIO e grava cada um como tabela Delta Lake no bucket `bronze`.

**Pré-requisitos:** Notebook `01` executado (CSVs no MinIO).

## 1. Imports e Configuração

In [1]:
import os
import boto3
from botocore.client import Config
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

print(f'MinIO: {MINIO_ENDPOINT}')
print(f'Landing: {LANDING_BUCKET} | Bronze: {BRONZE_BUCKET}')

MinIO: http://localhost:9020
Landing: landing-zone | Bronze: bronze


## 2. Criar SparkSession com Delta Lake e MinIO

In [2]:
spark = (
    SparkSession.builder
    .appName('CSV_to_Delta')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    # MinIO / S3A
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

your 131072x1 screen size is bogus. expect trouble
26/05/04 21:07:29 WARN Utils: Your hostname, DESKTOP-J27O5E0 resolves to a loopback address: 127.0.1.1; using 172.22.146.234 instead (on interface eth0)
26/05/04 21:07:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/william/spark-delta-minio-sqlserver/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/william/.ivy2/cache
The jars for the packages stored in: /home/william/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-357d28cc-fb55-4187-a013-049654bd7d2b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (205ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java

SparkSession criada com sucesso!


## 3. Criar Bucket Bronze no MinIO

In [3]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] ja existe')
except:
    s3_client.create_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] criado!')

print('Buckets:', [b['Name'] for b in s3_client.list_buckets()['Buckets']])

Bucket [bronze] criado!
Buckets: ['bronze', 'landing-zone']


## 4. Listar CSVs Disponíveis no Landing Zone

In [4]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
csv_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.csv')]

print(f'{len(csv_files)} arquivos CSV encontrados no bucket [{LANDING_BUCKET}]:')
for f in csv_files:
    print(f'  - {f}')

7 arquivos CSV encontrados no bucket [landing-zone]:
  - Agent_Policies.csv
  - Agents.csv
  - Claims.csv
  - Customers.csv
  - Insurance_Types.csv
  - Payments.csv
  - Policies.csv


## 5. Ler CSVs e Gravar como Delta Lake

In [5]:
from delta.tables import DeltaTable

print(f'Convertendo {len(csv_files)} CSVs para Delta Lake...\n')

for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    csv_path = f's3a://{LANDING_BUCKET}/{csv_file}'
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    
    # Ler CSV com inferência de schema
    df = spark.read \
        .option('header', 'true') \
        .option('inferSchema', 'true') \
        .csv(csv_path)
    
    # Gravar como Delta Lake
    df.write \
        .format('delta') \
        .mode('overwrite') \
        .save(delta_path)
    
    print(f'  {tabela}: {df.count()} registros | {len(df.columns)} colunas -> {delta_path}')

print(f'\nConversao concluida! {len(csv_files)} tabelas Delta criadas no bucket [{BRONZE_BUCKET}].')

Convertendo 7 CSVs para Delta Lake...



26/05/04 21:08:03 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


  Agent_Policies: 10 registros | 5 colunas -> s3a://bronze/Agent_Policies
  Agents: 10 registros | 5 colunas -> s3a://bronze/Agents
  Claims: 10 registros | 6 colunas -> s3a://bronze/Claims
  Customers: 10 registros | 7 colunas -> s3a://bronze/Customers
  Insurance_Types: 10 registros | 3 colunas -> s3a://bronze/Insurance_Types
  Payments: 10 registros | 5 colunas -> s3a://bronze/Payments
  Policies: 10 registros | 8 colunas -> s3a://bronze/Policies

Conversao concluida! 7 tabelas Delta criadas no bucket [bronze].


## 6. Validação - Ler Tabelas Delta

In [6]:
print('Validando tabelas Delta Lake...\n')

for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    
    # Verificar se é Delta
    is_delta = DeltaTable.isDeltaTable(spark, delta_path)
    df_delta = spark.read.format('delta').load(delta_path)
    
    print(f'  {tabela}: Delta={is_delta} | {df_delta.count()} registros | Colunas: {df_delta.columns}')

Validando tabelas Delta Lake...



26/05/04 21:08:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  Agent_Policies: Delta=True | 10 registros | Colunas: ['agent_policy_id', 'agent_id', 'policy_id', 'role', 'assigned_date']


  Agents: Delta=True | 10 registros | Colunas: ['agent_id', 'first_name', 'last_name', 'email', 'phone']


  Claims: Delta=True | 10 registros | Colunas: ['claim_id', 'policy_id', 'claim_date', 'claim_amount', 'claim_status', 'description']


  Customers: Delta=True | 10 registros | Colunas: ['customer_id', 'first_name', 'last_name', 'date_of_birth', 'email', 'phone', 'address']


  Insurance_Types: Delta=True | 10 registros | Colunas: ['insurance_type_id', 'type_name', 'description']


  Payments: Delta=True | 10 registros | Colunas: ['payment_id', 'policy_id', 'payment_date', 'amount', 'payment_method']


  Policies: Delta=True | 10 registros | Colunas: ['policy_id', 'customer_id', 'insurance_type_id', 'policy_number', 'start_date', 'end_date', 'premium_amount', 'policy_status']


In [9]:
# Amostra: exibir primeiros registros de algumas tabelas
for tabela in ['Customers', 'Agents', 'Claims']:
    print(f'\n--- {tabela.upper()} ---')
    spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/{tabela}').show(5)


--- CUSTOMERS ---
+-----------+----------+---------+-------------+--------------------+--------+------------+
|customer_id|first_name|last_name|date_of_birth|               email|   phone|     address|
+-----------+----------+---------+-------------+--------------------+--------+------------+
|          1|      John|      Doe|   1985-01-10|  john.doe@email.com|555-1111| 123 Main St|
|          2|      Jane|    Smith|   1990-02-20|jane.smith@email.com|555-1112|  456 Oak St|
|          3|    Carlos|    Silva|   1978-03-15|  carlos.s@email.com|555-1113| 789 Pine St|
|          4|      Anna|    Brown|   1988-04-18|    anna.b@email.com|555-1114|321 Maple St|
|          5|      Mike|   Wilson|   1992-05-22|    mike.w@email.com|555-1115|654 Cedar St|
+-----------+----------+---------+-------------+--------------------+--------+------------+
only showing top 5 rows


--- AGENTS ---
+--------+----------+---------+-----------------+--------+
|agent_id|first_name|last_name|            email|   p

In [10]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
